# Bioinf Project 7: Schistosoma mekongi

Финальный ноутбук для индивидуальной части проекта.

In [42]:
from pathlib import Path
from collections import defaultdict
import json
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")

In [43]:
def find_workspace_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "smekongi_dir").exists() and (path / "bioinf_project").exists():
            return path, path / "bioinf_project"
        if path.name == "bioinf_project" and (path.parent / "smekongi_dir").exists():
            return path.parent, path
    raise FileNotFoundError("Could not locate both smekongi_dir and bioinf_project")


WORKSPACE_ROOT, PROJECT_ROOT = find_workspace_root()
DATA_ROOT = WORKSPACE_ROOT / "smekongi_dir" / "ncbi_dataset" / "data" / "GCA_034768735.1"
RESULTS_ROOT = PROJECT_ROOT / "results" / "mekongi"
PLOTS_ROOT = RESULTS_ROOT / "plots"
TABLES_ROOT = RESULTS_ROOT / "structure_tables"
for p in (RESULTS_ROOT, PLOTS_ROOT, TABLES_ROOT):
    p.mkdir(parents=True, exist_ok=True)

FASTA = DATA_ROOT / "GCA_034768735.1_ASM3476873v1_genomic.fna"
GFF = DATA_ROOT / "genomic.gff"
ASSEMBLY_REPORT = DATA_ROOT.parent / "assembly_data_report.jsonl"
RAW_ZHUNT = DATA_ROOT / "zhunt_all.bed"
RAW_QUAD = DATA_ROOT / "quadruplexes.bed"
RAW_EPI = DATA_ROOT / "epigenetic_genes.csv"
RAW_DOMTBL = DATA_ROOT / "schistosoma_pfam.domtblout"

# Keep the results directory self-contained for the final repo.
for src, dst in [
    (RAW_ZHUNT, RESULTS_ROOT / "zhunt_all.bed"),
    (RAW_QUAD, RESULTS_ROOT / "quadruplexes.bed"),
    (RAW_EPI, RESULTS_ROOT / "epigenetic_genes.csv"),
    (RAW_DOMTBL, RESULTS_ROOT / "schistosoma_pfam.domtblout"),
]:
    if src.exists() and not dst.exists():
        shutil.copy2(src, dst)

print('WORKSPACE_ROOT =', WORKSPACE_ROOT)
print('PROJECT_ROOT   =', PROJECT_ROOT)
print('DATA_ROOT      =', DATA_ROOT)
print('RESULTS_ROOT   =', RESULTS_ROOT)

WORKSPACE_ROOT = /home/user/student_homeworks/bioinf/project
PROJECT_ROOT   = /home/user/student_homeworks/bioinf/project/bioinf_project
DATA_ROOT      = /home/user/student_homeworks/bioinf/project/smekongi_dir/ncbi_dataset/data/GCA_034768735.1
RESULTS_ROOT   = /home/user/student_homeworks/bioinf/project/bioinf_project/results/mekongi


In [44]:
from pathlib import Path
from collections import defaultdict
import json
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO
from IPython.display import display

sns.set_theme(style="whitegrid", context="talk")


def find_workspace_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / "smekongi_dir").exists() and (path / "bioinf_project").exists():
            return path, path / "bioinf_project"
        if path.name == "bioinf_project" and (path.parent / "smekongi_dir").exists():
            return path.parent, path
    raise FileNotFoundError("Could not locate both smekongi_dir and bioinf_project")


def parse_assembly_report(path):
    with path.open() as fh:
        obj = json.loads(next(fh))
    return {
        "accession": obj.get("accession"),
        "assembly_name": obj.get("assemblyInfo", {}).get("assemblyName"),
        "assembly_level": obj.get("assemblyInfo", {}).get("assemblyLevel"),
        "assembly_date": obj.get("assemblyInfo", {}).get("assemblyDate"),
        "annotation_name": obj.get("annotationInfo", {}).get("name"),
        "gene_count": obj.get("annotationInfo", {}).get("stats", {}).get("geneCounts", {}).get("total"),
    }


def parse_fasta_stats(path):
    lengths = {}
    chrom_lengths = {}
    chroms = []
    total_gc = 0
    total_len = 0
    all_lengths = []
    records = 0
    for rec in SeqIO.parse(str(path), "fasta"):
        seq = str(rec.seq).upper()
        L = len(seq)
        lengths[rec.id] = L
        all_lengths.append(L)
        records += 1
        total_len += L
        total_gc += seq.count("G") + seq.count("C")
        desc = rec.description.lower()
        if "chromosome" in desc and "unknown" not in desc:
            chroms.append(rec.id)
            chrom_lengths[rec.id] = L
    all_lengths.sort(reverse=True)
    half = sum(all_lengths) / 2
    csum = 0
    n50 = None
    l50 = None
    for i, L in enumerate(all_lengths, 1):
        csum += L
        if csum >= half:
            n50 = L
            l50 = i
            break
    gc_pct = 100 * total_gc / total_len if total_len else 0.0
    return {
        "lengths": lengths,
        "chrom_lengths": chrom_lengths,
        "chroms": chroms,
        "records": records,
        "genome_bp": total_len,
        "gc_pct": gc_pct,
        "n50": n50,
        "l50": l50,
    }


def read_bed_df(path):
    rows = []
    with path.open() as fh:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 3:
                continue
            score = None
            try:
                score = float(f[-1])
            except ValueError:
                score = None
            rows.append({
                "chrom": f[0],
                "start": int(f[1]),
                "end": int(f[2]),
                "name": f[3] if len(f) >= 4 else ".",
                "score": score,
                "n_fields": len(f),
            })
    return pd.DataFrame(rows)


def bed_by_chrom(path):
    by = defaultdict(list)
    with path.open() as fh:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 3:
                continue
            by[f[0]].append((int(f[1]), int(f[2])))
    for chrom in by:
        by[chrom].sort(key=lambda x: (x[0], x[1]))
    return dict(by)


def merge_intervals(intervals):
    cleaned = sorted((int(s), int(e)) for s, e in intervals if int(e) > int(s))
    merged = []
    for s, e in cleaned:
        if not merged or s > merged[-1][1]:
            merged.append([s, e])
        else:
            merged[-1][1] = max(merged[-1][1], e)
    return [(s, e) for s, e in merged]


def merge_by_chrom(intervals_by_chrom):
    return {chrom: merge_intervals(intervals) for chrom, intervals in intervals_by_chrom.items()}


def total_bp(intervals_by_chrom):
    return sum(e - s for intervals in intervals_by_chrom.values() for s, e in merge_intervals(intervals))


def hit_count(source_by_chrom, target_by_chrom):
    n = 0
    for chrom, source in source_by_chrom.items():
        target = target_by_chrom.get(chrom, [])
        j = 0
        for s, e in source:
            while j < len(target) and target[j][1] <= s:
                j += 1
            if j < len(target) and target[j][0] < e:
                n += 1
    return n


def intersection_bp(a_by_chrom, b_by_chrom):
    tot = 0
    for chrom in sorted(set(a_by_chrom) | set(b_by_chrom)):
        a = merge_intervals(a_by_chrom.get(chrom, []))
        b = merge_intervals(b_by_chrom.get(chrom, []))
        i = j = 0
        while i < len(a) and j < len(b):
            s = max(a[i][0], b[j][0])
            e = min(a[i][1], b[j][1])
            if e > s:
                tot += e - s
            if a[i][1] <= b[j][1]:
                i += 1
            else:
                j += 1
    return tot


def subtract_intervals(base, subtract):
    out = []
    j = 0
    for s, e in base:
        cur = s
        while j < len(subtract) and subtract[j][1] <= cur:
            j += 1
        k = j
        while k < len(subtract) and subtract[k][0] < e:
            ss, ee = subtract[k]
            if ss > cur:
                out.append((cur, min(ss, e)))
            cur = max(cur, ee)
            if cur >= e:
                break
            k += 1
        if cur < e:
            out.append((cur, e))
    return out


def complement_intervals(base_by_chrom, chrom_lengths):
    out = {}
    for chrom, length in chrom_lengths.items():
        base = merge_intervals(base_by_chrom.get(chrom, []))
        cur = 0
        comp = []
        for s, e in base:
            if cur < s:
                comp.append((cur, s))
            cur = max(cur, e)
        if cur < length:
            comp.append((cur, length))
        out[chrom] = comp
    return out


def parse_gff_regions(path, chroms, chrom_lengths):
    genes = defaultdict(list)
    exons = defaultdict(list)
    stranded_genes = defaultdict(list)
    chrom_set = set(chroms)
    gene_count = 0
    with path.open() as fh:
        for line in fh:
            if not line.strip() or line.startswith("#"):
                continue
            f = line.rstrip("\n").split("\t")
            if len(f) < 9 or f[0] not in chrom_set:
                continue
            chrom, feature = f[0], f[2]
            try:
                start = int(f[3]) - 1
                end = int(f[4])
            except ValueError:
                continue
            strand = f[6] if f[6] in ("+", "-") else "+"
            if feature == "gene":
                genes[chrom].append((start, end))
                stranded_genes[chrom].append((start, end, strand))
                gene_count += 1
            elif feature == "exon":
                exons[chrom].append((start, end))
    genes = merge_by_chrom(genes)
    exons = merge_by_chrom(exons)
    promoters = defaultdict(list)
    downstream = defaultdict(list)
    for chrom, lst in stranded_genes.items():
        L = chrom_lengths[chrom]
        for start, end, strand in lst:
            if strand == "+":
                promoters[chrom].append((max(0, start - 1000), start))
                downstream[chrom].append((end, min(L, end + 200)))
            else:
                promoters[chrom].append((end, min(L, end + 1000)))
                downstream[chrom].append((max(0, start - 200), start))
    promoters = merge_by_chrom(promoters)
    downstream = merge_by_chrom(downstream)
    introns = {chrom: merge_intervals(subtract_intervals(genes.get(chrom, []), exons.get(chrom, []))) for chrom in chroms}
    intergenic = complement_intervals(genes, chrom_lengths)
    return {
        "Exons": exons,
        "Introns": introns,
        "Promoters (1000 up from TSS)": promoters,
        "Downstream (200 bp)": downstream,
        "Intergenic": intergenic,
        "gene_count": gene_count,
    }


def validate_bed_df(df, chrom_lengths=None):
    rows = []
    if df.empty:
        return pd.DataFrame(rows)
    for chrom, g in df.groupby("chrom", sort=False):
        starts = g["start"].tolist()
        ends = g["end"].tolist()
        sorted_ok = all(starts[i] <= starts[i + 1] for i in range(len(starts) - 1))
        in_bounds = True
        if chrom_lengths and chrom in chrom_lengths:
            L = chrom_lengths[chrom]
            in_bounds = all(0 <= s < e <= L for s, e in zip(starts, ends))
        score_vals = g["score"].dropna()
        rows.append({
            "chrom": chrom,
            "n_intervals": len(g),
            "starts_sorted": bool(sorted_ok),
            "in_bounds": bool(in_bounds),
            "n_fields_mode": int(g["n_fields"].mode().iat[0]) if not g["n_fields"].mode().empty else None,
            "score_min": float(score_vals.min()) if not score_vals.empty else None,
            "score_max": float(score_vals.max()) if not score_vals.empty else None,
        })
    return pd.DataFrame(rows)


def interval_summary(source_by_chrom, target_by_chrom, chrom_order):
    rows = []
    for chrom in chrom_order:
        source = merge_intervals(source_by_chrom.get(chrom, []))
        target = merge_intervals(target_by_chrom.get(chrom, []))
        source_dict = {chrom: source}
        target_dict = {chrom: target}
        src_n = len(source)
        tgt_n = len(target)
        src_bp = sum(e - s for s, e in source)
        tgt_bp = sum(e - s for s, e in target)
        hit_src = hit_count(source_dict, target_dict)
        hit_tgt = hit_count(target_dict, source_dict)
        ov_bp = intersection_bp(source_dict, target_dict)
        union_bp = src_bp + tgt_bp - ov_bp
        rows.append({
            "chromosome": chrom,
            "source_intervals": src_n,
            "target_intervals": tgt_n,
            "source_bp": src_bp,
            "target_bp": tgt_bp,
            "source_intervals_hit": hit_src,
            "target_intervals_hit": hit_tgt,
            "overlap_bp": ov_bp,
            "jaccard": ov_bp / union_bp if union_bp else 0.0,
            "target_inside_source": bool(hit_tgt == tgt_n and ov_bp == tgt_bp),
            "source_inside_target": bool(hit_src == src_n and ov_bp == src_bp),
        })
    source = merge_intervals([iv for chrom in chrom_order for iv in source_by_chrom.get(chrom, [])])
    target = merge_intervals([iv for chrom in chrom_order for iv in target_by_chrom.get(chrom, [])])
    src_n = len(source)
    tgt_n = len(target)
    src_bp = sum(e - s for s, e in source)
    tgt_bp = sum(e - s for s, e in target)
    source_dict = {"Genome": source}
    target_dict = {"Genome": target}
    hit_src = hit_count(source_dict, target_dict)
    hit_tgt = hit_count(target_dict, source_dict)
    ov_bp = intersection_bp(source_dict, target_dict)
    union_bp = src_bp + tgt_bp - ov_bp
    rows.append({
        "chromosome": "Genome",
        "source_intervals": src_n,
        "target_intervals": tgt_n,
        "source_bp": src_bp,
        "target_bp": tgt_bp,
        "source_intervals_hit": hit_src,
        "target_intervals_hit": hit_tgt,
        "overlap_bp": ov_bp,
        "jaccard": ov_bp / union_bp if union_bp else 0.0,
        "target_inside_source": bool(hit_tgt == tgt_n and ov_bp == tgt_bp),
        "source_inside_target": bool(hit_src == src_n and ov_bp == src_bp),
    })
    return pd.DataFrame(rows)


def build_structure_tables(structure_sets, feature_sets, chrom_lengths, outdir):
    genome_bp = sum(chrom_lengths.values())
    category_order = [
        "Exons",
        "Introns",
        "Promoters (1000 up from TSS)",
        "Downstream (200 bp)",
        "Intergenic",
    ]
    rows1 = []
    rows2 = []
    for cat in category_order:
        feature = feature_sets[cat]
        feature_bp = sum(e - s for intervals in feature.values() for s, e in intervals)
        bg = feature_bp / genome_bp if genome_bp else 0.0
        row1 = {
            "Region": cat,
            "region_bp": feature_bp,
            "bg_fraction": round(bg, 6),
            "n_regions_total": sum(len(v) for v in feature.values()),
        }
        row2 = {
            "Region": cat,
            "n_regions_total": sum(len(v) for v in feature.values()),
        }
        for name, struct in structure_sets.items():
            hit_struct = hit_count(struct, feature)
            hit_feat = hit_count(feature, struct)
            total_struct = sum(len(v) for v in struct.values())
            row1[f"{name}_count"] = hit_struct
            row1[f"{name}_fraction"] = round(hit_struct / total_struct, 4) if total_struct else 0.0
            row1[f"{name}_enrichment_vs_bg"] = round((hit_struct / total_struct) / bg, 2) if bg > 0 and total_struct else 0.0
            row2[f"{name}_n_regions_hit"] = hit_feat
            row2[f"{name}_frac_regions_hit"] = round(hit_feat / row2["n_regions_total"], 4) if row2["n_regions_total"] else 0.0
        rows1.append(row1)
        rows2.append(row2)
    table1 = pd.DataFrame(rows1)
    table2 = pd.DataFrame(rows2)
    outdir.mkdir(parents=True, exist_ok=True)
    table1.to_csv(outdir / "table1.csv", index=False)
    table2.to_csv(outdir / "table2.csv", index=False)
    return table1, table2

In [45]:
assembly = parse_assembly_report(ASSEMBLY_REPORT)
fasta_stats = parse_fasta_stats(FASTA)
chrom_order = fasta_stats['chroms']
chrom_lengths = fasta_stats['chrom_lengths']

print('Assembly stats:')
display(pd.DataFrame([
    {'metric': 'accession', 'value': assembly['accession']},
    {'metric': 'assembly_name', 'value': assembly['assembly_name']},
    {'metric': 'assembly_level', 'value': assembly['assembly_level']},
    {'metric': 'assembly_date', 'value': assembly['assembly_date']},
    {'metric': 'genome_bp', 'value': fasta_stats['genome_bp']},
    {'metric': 'gc_pct', 'value': round(fasta_stats['gc_pct'], 3)},
    {'metric': 'n50', 'value': fasta_stats['n50']},
    {'metric': 'l50', 'value': fasta_stats['l50']},
    {'metric': 'records', 'value': fasta_stats['records']},
    {'metric': 'chromosomes', 'value': len(chrom_order)},
]))

manifest = pd.DataFrame([
    {'label': 'FASTA', 'path': str(FASTA), 'exists': FASTA.exists(), 'size_mb': round(FASTA.stat().st_size / 1024 / 1024, 3)},
    {'label': 'GFF', 'path': str(GFF), 'exists': GFF.exists(), 'size_mb': round(GFF.stat().st_size / 1024 / 1024, 3)},
    {'label': 'zhunt_all.bed', 'path': str(RESULTS_ROOT / 'zhunt_all.bed'), 'exists': (RESULTS_ROOT / 'zhunt_all.bed').exists(), 'size_mb': round((RESULTS_ROOT / 'zhunt_all.bed').stat().st_size / 1024 / 1024, 3)},
    {'label': 'quadruplexes.bed', 'path': str(RESULTS_ROOT / 'quadruplexes.bed'), 'exists': (RESULTS_ROOT / 'quadruplexes.bed').exists(), 'size_mb': round((RESULTS_ROOT / 'quadruplexes.bed').stat().st_size / 1024 / 1024, 3)},
    {'label': 'epigenetic_genes.csv', 'path': str(RESULTS_ROOT / 'epigenetic_genes.csv'), 'exists': (RESULTS_ROOT / 'epigenetic_genes.csv').exists(), 'size_mb': round((RESULTS_ROOT / 'epigenetic_genes.csv').stat().st_size / 1024 / 1024, 3)},
    {'label': 'schistosoma_pfam.domtblout', 'path': str(RESULTS_ROOT / 'schistosoma_pfam.domtblout'), 'exists': (RESULTS_ROOT / 'schistosoma_pfam.domtblout').exists(), 'size_mb': round((RESULTS_ROOT / 'schistosoma_pfam.domtblout').stat().st_size / 1024 / 1024, 3)},
])
manifest.to_csv(RESULTS_ROOT / 'input_manifest.csv', index=False)
display(manifest)

Assembly stats:


,metric,value
0,accession,GCA_034768735.1
1,assembly_name,ASM3476873v1
2,assembly_level,Chromosome
3,assembly_date,None
4,genome_bp,404245338
5,gc_pct,34.597
6,n50,46759691
7,l50,3
8,records,1279
9,chromosomes,8


,label,path,exists,size_mb
0,FASTA,/home/user/student_homeworks/bioinf/project/sm...,True,390.464
1,GFF,/home/user/student_homeworks/bioinf/project/sm...,True,38.980
2,zhunt_all.bed,/home/user/student_homeworks/bioinf/project/bi...,True,1.447
3,quadruplexes.bed,/home/user/student_homeworks/bioinf/project/bi...,True,0.085
4,epigenetic_genes.csv,/home/user/student_homeworks/bioinf/project/bi...,True,0.015
5,schistosoma_pfam.domtblout,/home/user/student_homeworks/bioinf/project/bi...,True,7.236


## 1. Проверка `ZDNABERT`

In [46]:

zd_parts = sorted(DATA_ROOT.glob('zdnabert_*.bed'))
assert len(zd_parts) == len(chrom_order), f'Expected {len(chrom_order)} ZDNABERT files, found {len(zd_parts)}'

zd_rows = []
zd_all = RESULTS_ROOT / 'zdnabert_all.bed'
with zd_all.open('w') as out:
    for part in zd_parts:
        with part.open() as fh:
            for line in fh:
                if line.strip():
                    out.write(line if line.endswith('\n') else line + '\n')

for part in zd_parts:
    df = read_bed_df(part)
    chrom = part.stem.split('_', 1)[1]
    L = chrom_lengths[chrom]
    starts_sorted = bool((df['start'].diff().fillna(0) >= 0).all())
    in_bounds = bool(((df['start'] >= 0) & (df['end'] <= L) & (df['end'] > df['start'])).all())
    score_vals = df['score'].dropna()
    zd_rows.append({
        'chromosome': chrom,
        'file': part.name,
        'n_intervals': int(len(df)),
        'starts_sorted': starts_sorted,
        'in_bounds': in_bounds,
        'n_fields_mode': int(df['n_fields'].mode().iat[0]) if not df['n_fields'].mode().empty else None,
        'score_min': float(score_vals.min()) if not score_vals.empty else None,
        'score_max': float(score_vals.max()) if not score_vals.empty else None,
    })

zd_validation = pd.DataFrame(zd_rows)
zd_validation.to_csv(RESULTS_ROOT / 'zdnabert_validation.csv', index=False)
display(zd_validation)
assert zd_validation['starts_sorted'].all()
assert zd_validation['in_bounds'].all()
print('Combined ZDNABERT BED:', zd_all)
print('Total intervals:', sum(zd_validation['n_intervals']))


,chromosome,file,n_intervals,starts_sorted,in_bounds,n_fields_mode,score_min,score_max
0,CM068305.1,zdnabert_CM068305.1.bed,1296,True,True,5,0.6018,0.9817
1,CM068306.1,zdnabert_CM068306.1.bed,1384,True,True,5,0.5948,0.9818
2,CM068307.1,zdnabert_CM068307.1.bed,638,True,True,5,0.6296,0.9792
3,CM068308.1,zdnabert_CM068308.1.bed,631,True,True,5,0.6249,0.9788
4,CM068309.1,zdnabert_CM068309.1.bed,647,True,True,5,0.5864,0.9784
5,CM068310.1,zdnabert_CM068310.1.bed,355,True,True,5,0.5657,0.9809
6,CM068311.1,zdnabert_CM068311.1.bed,403,True,True,5,0.5741,0.9823
7,CM068312.1,zdnabert_CM068312.1.bed,270,True,True,5,0.6371,0.9738


Combined ZDNABERT BED: /home/user/student_homeworks/bioinf/project/bioinf_project/results/mekongi/zdnabert_all.bed
Total intervals: 5624


## 2. `zhunt` vs `ZDNABERT`

In [1]:
%%bash

WORKSPACE="/home/user/student_homeworks/bioinf/project"
RESULTS_ROOT="$WORKSPACE/bioinf_project/results/mekongi"
DATA_ROOT="$WORKSPACE/smekongi_dir/ncbi_dataset/data/GCA_034768735.1"
sort -k1,1 -k2,2n "$RESULTS_ROOT/zhunt_all_without_merge.bed" \
    | bedtools merge -i - -c 5 -o max > "$RESULTS_ROOT/zhunt_all.bed"

## 3. Эпигенетические гены

In [48]:
epi = pd.read_csv(RESULTS_ROOT / 'epigenetic_genes.csv')
family_counts = epi['Семейство'].value_counts().reset_index()
family_counts.columns = ['Семейство', 'count']
family_counts.to_csv(RESULTS_ROOT / 'epigenetic_family_counts.csv', index=False)
display(epi.head())
display(family_counts)

fig, ax = plt.subplots(figsize=(10, max(4, 0.28 * epi['Семейство'].nunique())))
family_counts.sort_values('count').set_index('Семейство')['count'].plot(kind='barh', color='#5b8ff9', ax=ax)
ax.set_xlabel('Number of hits')
ax.set_ylabel('Epigenetic family')
ax.set_title('Epigenetic families detected in the proteome')
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'epigenetic_families.png', dpi=220)
plt.close(fig)

,Семейство,Pfam,Ген,protein_id,chrom,start,end,evalue
0,Acetyltransferase (GNAT),PF00583 (Acetyltransf_1),KAK4469646.1,KAK4469646.1,CM068309.1,29742299,29743991,1.100000e-16
1,Acetyltransferase (GNAT),PF00583 (Acetyltransf_1),KAK4470955.1,KAK4470955.1,CM068308.1,40411637,40412301,3.000000e-15
2,Acetyltransferase (GNAT),PF00583 (Acetyltransf_1),KAK4473755.1,KAK4473755.1,CM068306.1,27960285,27960906,5.800000e-14
3,Acetyltransferase (GNAT),PF00583 (Acetyltransf_1),KAK4471523.1,KAK4471523.1,CM068307.1,17456212,17456901,4.900000e-12
4,Acetyltransferase (GNAT),PF00583 (Acetyltransf_1),KAK4474981.1,KAK4474981.1,CM068305.1,66319328,66341981,3.000000e-11


,Семейство,count
0,PHD finger,30
1,SNF2_N (chromatin remodeler ATPase),22
2,SET (histone lysine methyltransferase),19
3,Bromodomain,16
4,Chromo domain,9
5,Acetyltransferase (GNAT),8
6,JmjC (histone demethylase),8
7,Histone deacetylase (HDAC),7
8,PWWP,6
9,SIR2 (NAD-dep. deacetylase),5


## 4. Таблицы по регионам генома

Здесь считаем пересечения для `quadruplexes`, `zhunt` и `ZDNABERT` по
пяти категориям: `Exons`, `Introns`, `Promoters`, `Downstream`, `Intergenic`.

In [49]:
feature_sets = parse_gff_regions(GFF, chrom_order, chrom_lengths)
gene_count = feature_sets.pop('gene_count')
quad_by = bed_by_chrom(RESULTS_ROOT / 'quadruplexes.bed')
struct_sets = {
    'quadruplex': quad_by,
    'zhunt': zhunt_by,
    'zdnabert': zd_by,
}
table1, table2 = build_structure_tables(struct_sets, feature_sets, chrom_lengths, TABLES_ROOT)
table1.to_csv(TABLES_ROOT / 'table1.csv', index=False)
table2.to_csv(TABLES_ROOT / 'table2.csv', index=False)
display(table1)
display(table2)

,Region,region_bp,bg_fraction,n_regions_total,quadruplex_count,quadruplex_fraction,quadruplex_enrichment_vs_bg,zhunt_count,zhunt_fraction,zhunt_enrichment_vs_bg,zdnabert_count,zdnabert_fraction,zdnabert_enrichment_vs_bg
0,Exons,16433107,0.043360,68083,16,0.0054,0.13,1167,0.0287,0.66,91,0.0162,0.37
1,Introns,161993417,0.427431,59371,1065,0.3622,0.85,17513,0.4314,1.01,1926,0.3425,0.80
2,Promoters (1000 up from TSS),8672480,0.022883,8591,74,0.0252,1.10,1075,0.0265,1.16,108,0.0192,0.84
3,Downstream (200 bp),1737855,0.004585,8650,22,0.0075,1.63,240,0.0059,1.29,6,0.0011,0.23
4,Intergenic,200566864,0.529209,8720,1860,0.6327,1.20,22131,0.5451,1.03,3609,0.6417,1.21


,Region,n_regions_total,quadruplex_n_regions_hit,quadruplex_frac_regions_hit,zhunt_n_regions_hit,zhunt_frac_regions_hit,zdnabert_n_regions_hit,zdnabert_frac_regions_hit
0,Exons,68083,16,0.0002,1139,0.0167,89,0.0013
1,Introns,59371,950,0.0160,11595,0.1953,1592,0.0268
2,Promoters (1000 up from TSS),8591,68,0.0079,969,0.1128,97,0.0113
3,Downstream (200 bp),8650,22,0.0025,235,0.0272,6,0.0007
4,Intergenic,8720,1108,0.1271,5898,0.6764,1907,0.2187


## 5. Графики и summary

In [50]:

fig, ax = plt.subplots(figsize=(14, 6))
count_cols = [c for c in table1.columns if c.endswith('_count')]
sns.heatmap(table1.set_index('Region')[count_cols], annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Counts of predicted structures by genomic region')
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'structure_counts_heatmap.png', dpi=220)
plt.close(fig)

fig, ax = plt.subplots(figsize=(14, 6))
frac_cols = [c for c in table1.columns if c.endswith('_fraction')]
sns.heatmap(table1.set_index('Region')[frac_cols], annot=True, fmt='.2f', cmap='Oranges', ax=ax)
ax.set_title('Fraction of each structure class in genomic regions')
fig.tight_layout()
fig.savefig(PLOTS_ROOT / 'structure_fraction_heatmap.png', dpi=220)
plt.close(fig)

summary = {
    'accession': assembly['accession'],
    'assembly_name': assembly['assembly_name'],
    'assembly_level': assembly['assembly_level'],
    'assembly_date': assembly['assembly_date'],
    'annotation_name': assembly['annotation_name'],
    'genome_bp': fasta_stats['genome_bp'],
    'gc_pct': round(fasta_stats['gc_pct'], 3),
    'n50': fasta_stats['n50'],
    'l50': fasta_stats['l50'],
    'records': fasta_stats['records'],
    'chromosomes': len(chrom_order),
    'chromosome_ids': chrom_order,
    'gene_count': int(assembly['gene_count'] or gene_count),
    'epigenetic_hits': int(len(epi)),
    'epigenetic_families': int(epi['Семейство'].nunique()),
    'zhunt_intervals': int(sum(len(v) for v in zhunt_by.values())),
    'zdnabert_intervals': int(sum(len(v) for v in zd_by.values())),
    'quadruplex_intervals': int(sum(len(v) for v in quad_by.values())),
    'zhunt_bp': int(total_bp(zhunt_by)),
    'zdnabert_bp': int(total_bp(zd_by)),
    'quadruplex_bp': int(total_bp(quad_by)),
    'zhunt_vs_zdnabert_overlap_bp': int(intersection_bp(zhunt_by, zd_by)),
    'zhunt_vs_zdnabert_jaccard': round(
        intersection_bp(zhunt_by, zd_by) / (total_bp(zhunt_by) + total_bp(zd_by) - intersection_bp(zhunt_by, zd_by)),
        6,
    ) if (total_bp(zhunt_by) + total_bp(zd_by) - intersection_bp(zhunt_by, zd_by)) else 0.0,
    'zdnabert_inside_zhunt': bool(hit_count(zd_by, zhunt_by) == sum(len(v) for v in zd_by.values())),
}
(RESULTS_ROOT / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2) + '\\n', encoding='utf-8')

structure_summary = []
for name, struct in struct_sets.items():
    structure_summary.append({
        'structure': name,
        'intervals': sum(len(v) for v in struct.values()),
        'covered_bp': total_bp(struct),
        'chromosomes': len(struct),
    })
pd.DataFrame(structure_summary).to_csv(RESULTS_ROOT / 'structure_summary.csv', index=False)

display(pd.DataFrame([summary]))
print('Saved summary.json and plots to:', RESULTS_ROOT)


,accession,assembly_name,assembly_level,assembly_date,annotation_name,genome_bp,gc_pct,n50,l50,records,...,epigenetic_families,zhunt_intervals,zdnabert_intervals,quadruplex_intervals,zhunt_bp,zdnabert_bp,quadruplex_bp,zhunt_vs_zdnabert_overlap_bp,zhunt_vs_zdnabert_jaccard,zdnabert_inside_zhunt
0,GCA_034768735.1,ASM3476873v1,Chromosome,None,Annotation submitted by Sun Yat-sen University,404245338,34.597,46759691,3,1279,...,18,40597,5624,2940,1002503,101968,143979,23872,0.022091,False


Saved summary.json and plots to: /home/user/student_homeworks/bioinf/project/bioinf_project/results/mekongi
